# SpaceX Falcon 9 - Data Wrangling

Starting from the raw web-scraped launch table, this notebook cleans it up and builds the binary landing-success label (`Class`) that everything downstream — EDA, SQL, and the classification models — depends on.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('spacex_web_scraped.csv')
df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success\n,F9 v1.0B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.0B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.0B0005.1,No attempt\n,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success\n,F9 v1.0B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success\n,F9 v1.0B0007.1,No attempt\n,1 March 2013,15:10


### Clean the payload mass column

`Payload mass` came in as text (`"4,700 kg"`, or `"0"` for a few early flights). Strip the commas and units, convert to numeric, and fill any missing values with the column mean.

In [2]:
df['Payload mass'] = (
    df['Payload mass']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' kg', '', regex=False)
)
df['Payload mass'] = pd.to_numeric(df['Payload mass'], errors='coerce')

missing_before = df['Payload mass'].isnull().sum()
mean_payload = df['Payload mass'].mean()
df['Payload mass'] = df['Payload mass'].fillna(mean_payload)

print(f"Missing payload values before fill: {missing_before}")
print(f"Filled with column mean: {mean_payload:.1f} kg")

Missing payload values before fill: 8
Filled with column mean: 6191.1 kg


### Clean the launch outcome and booster landing columns

Both came with stray whitespace/newlines from the scrape. Strip them down to clean category labels.

In [3]:
df['Launch outcome'] = df['Launch outcome'].astype(str).str.strip()
df['Booster landing'] = df['Booster landing'].astype(str).str.strip()

df['Booster landing'].value_counts()

Booster landing
Success         65
No attempt      22
Failure         10
Controlled       5
Uncontrolled     2
Precluded        1
0                1
Name: count, dtype: int64

### Build the target label: Class

A landing counts as successful (`Class = 1`) if the booster landing outcome was `Success` or `Controlled` (a controlled ocean landing, generally an early deliberate soft-splashdown test). Everything else — `Failure`, `Uncontrolled`, `No attempt`, `Precluded`, or a blank `0` — counts as unsuccessful (`Class = 0`).

In [4]:
successful_outcomes = {'Success', 'Controlled'}
df['Class'] = df['Booster landing'].apply(lambda x: 1 if x in successful_outcomes else 0)

print("Overall landing success rate: {:.1%}".format(df['Class'].mean()))
df[['Flight No.', 'Launch site', 'Payload mass', 'Booster landing', 'Class']].head(10)

Overall landing success rate: 66.0%


,Flight No.,Launch site,Payload mass,Booster landing,Class
0,1,CCAFS,0.0,Failure,0
1,2,CCAFS,0.0,Failure,0
2,3,CCAFS,525.0,No attempt,0
3,4,CCAFS,4700.0,No attempt,0
4,5,CCAFS,4877.0,No attempt,0
5,6,VAFB,500.0,Uncontrolled,0
6,7,CCAFS,3170.0,No attempt,0
7,8,CCAFS,3325.0,No attempt,0
8,9,Cape Canaveral,2296.0,Controlled,1
9,10,Cape Canaveral,1316.0,Controlled,1


### Check for any remaining missing values across the wrangled table

In [5]:
df.isnull().sum()

Flight No.         0
Launch site        0
Payload            0
Payload mass       0
Orbit              0
Customer           1
Launch outcome     0
Version Booster    0
Booster landing    0
Date               0
Time               0
Class              0
dtype: int64

### Save the wrangled dataset

In [6]:
df.to_csv('dataset_wrangled.csv', index=False)
print("Saved", df.shape[0], "rows and", df.shape[1], "columns to dataset_wrangled.csv")

Saved 106 rows and 12 columns to dataset_wrangled.csv


## Summary

The raw scraped table had text-formatted payload masses, a few missing values, and inconsistent whitespace in the outcome columns. After cleaning, every launch has a numeric payload mass and a clean binary `Class` label, which downstream notebooks (EDA, SQL, Folium, Dash, and the classification models) all build on directly.